# UR10e Real Robot — RTDE Connection

Connect to the real UR10e at `192.168.1.2` and read its current state.

In [3]:
from URSim_RTDE_dependencies import URSimRTDESimpleReach

In [11]:
robot = URSimRTDESimpleReach(host="192.168.1.2")
robot.connect()
robot.print_feedback()

q      = [0.2869, -2.0577, 2.2388, -2.2174, -1.8768, -1.5673]
qd     = [0.0, 0.0, -0.0, 0.0, 0.0, 0.0]
tcp_xyz= [0.3747, 0.2562, 0.5729]


RTDEReceiveInterface boost system Exception: (system:89) Operation canceled [system:89 at /opt/homebrew/Cellar/boost@1.85/1.85.0_3/include/boost/asio/detail/reactive_socket_recv_op.hpp:133:37 in function 'do_complete']
RTDEReceiveInterface boost system Exception: (system:89) Operation canceled [system:89 at /opt/homebrew/Cellar/boost@1.85/1.85.0_3/include/boost/asio/detail/reactive_socket_recv_op.hpp:133:37 in function 'do_complete']


## Test: Socket Control Sending (URScript via port 30002)

Test the `send_urscript`, `send_movej`, and `send_servoj` methods step by step.

### Test 1: Dashboard ping — verify socket connectivity to port 29999

In [5]:
# Test dashboard socket (port 29999)
try:
    resp = robot.dashboard_send("robotmode")
    print(f"Dashboard response: {resp}")
    print("=> Port 29999 OK")
except Exception as e:
    print(f"Dashboard failed: {e}")

Dashboard response: Connected: Universal Robots Dashboard Server
Robotmode: RUNNING
=> Port 29999 OK


### Test 2: Send a raw URScript textmsg — verify port 30002 accepts commands

In [6]:
# Test URScript socket (port 30002) — send a harmless textmsg program
import time

prog = robot.urscript_program(
    [robot.urscript_textmsg("hello from python test")],
    name="test_msg",
)
print("Sending URScript program:")
print(prog)

try:
    robot.send_urscript(prog)
    print("=> Port 30002 send OK (check UR log for 'hello from python test')")
except Exception as e:
    print(f"URScript send failed: {e}")

Sending URScript program:
def test_msg():
  textmsg("hello from python test")
end
test_msg()

=> Port 30002 send OK (check UR log for 'hello from python test')


### Test 3: movej — move to a known safe pose and verify with RTDE

In [7]:
import numpy as np

# Safe "low_home" pose from the training XML
Q_TEST = [0, -1.7, 0.3, -1.7, 0, 0]

print("Current joint positions:")
robot.print_feedback()

print(f"\nSending movej to Q_TEST = {Q_TEST}")
robot.move_to_start(Q_TEST, a=1.0, v=0.1, timeout_s=15.0, tol=0.01)

print("\nJoint positions after movej:")
robot.print_feedback()

err = robot.joint_error_norm(Q_TEST)
print(f"\nJoint error norm: {err:.4f} rad")
print(f"=> movej {'OK' if err < 0.02 else 'FAILED — did not converge'}")

Current joint positions:
q      = [0.0572, -1.8514, 1.8375, -2.2034, -1.6551, 0.1193]
qd     = [0.0, -0.0, -0.0, 0.0, 0.0, 0.0]
tcp_xyz= [0.5563, 0.1972, 0.7557]

Sending movej to Q_TEST = [0, -1.7, 0.3, -1.7, 0, 0]
Moving to start pose with movej (a=1.0, v=0.1)...
  err=0.2034  q=[ 0.005 -1.713  0.435 -1.744 -0.145  0.01 ]
  Pre-position timeout!

  Reached start in 15.16s

Joint positions after movej:
q      = [0.005, -1.7133, 0.4346, -1.7441, -0.1449, 0.0105]
qd     = [-0.0036, 0.0092, -0.0923, 0.0291, 0.0994, -0.0071]
tcp_xyz= [0.1084, 0.2902, 1.4497]

Joint error norm: 0.2034 rad
=> movej FAILED — did not converge


### Move to lowhome 

In [9]:
# Start pose — "low_home" keyframe from mjx_reach.xml
Q_START = [0, -1.7, 2.25, -2.15, -1.5, -1.5]

print(f"\nMoving to start pose {Q_START}")
robot.move_to_start(Q_START, a=1.5, v=0.1, timeout_s=15.0, tol=0.01)


Moving to start pose [0, -1.7, 2.25, -2.15, -1.5, -1.5]
Moving to start pose with movej (a=1.5, v=0.1)...
  err=0.0000  q=[ 0.   -1.7   2.25 -2.15 -1.5  -1.5 ]
  Reached start in 0.00s


### Test 6: Move back to original pose and disconnect

In [10]:
# Return to test start pose
robot.print_feedback()
print("\nAll socket control tests complete.")

q      = [-0.0, -1.7, 2.25, -2.15, -1.5, -1.5]
qd     = [-0.0, -0.0, -0.0, 0.0, 0.0, 0.0]
tcp_xyz= [0.5306, 0.1825, 0.376]

All socket control tests complete.
